### Molecular Geometry + Post-Processing Algorithms
#### This notebook is essentially combining the Post-Processing Algorithms notebook and Max, Adelly, and I (Linda)'s work on the molecular geometry bond length coordinates!

In [1]:
from math import sin, cos, tan, sqrt, pow, pi, asin, acos, atan

In [2]:
from typing import List
from typing_extensions import DefaultDict
from collections import deque
import random

In [3]:
import numpy as np
np.random.seed(69420)

In [4]:
from urllib.request import urlopen
from urllib.parse import quote

In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem

In [6]:
import numpy as np
%run MathFunc.py

### Useful Tools

In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('../DFS/bondlengths.csv', encoding='utf-8')

In [9]:
B = list(df.to_dict()['bonds'].values())
L = list(df.to_dict()['lengths'].values())

In [10]:
BondLengths = {B[i] : L[i] for i in range(len(df))}

In [11]:
GeometryMap = {
    (2, 0): "Linear",
    (2, 1): "Bent21",
    (2, 2): "Bent22",
    (3, 0): "Trigonal-Planar",
    (3, 1): "Trigonal-Pyramidal",
    (4, 0): "Tetrahedral",
    (5, 0): "Trigonal-Bipyramidal",
    (4, 1): "Seesaw",
    (3, 2): "T-Shaped",
    (2, 3): "Linear",
    (6, 0): "Octahedral",
    (5, 1): "Square-Pyramidal",
    (4, 2): "Square-Planar",
    #(Bond Pairs, Lone Pairs); no expanded octets for now
}

In [12]:
bondType = {1: "-", 2: "=", 3: "#",} 

In [13]:
def DegtoRad(theta: float) -> float:
    #Python trigonometric functions use Radians
    return theta * pi / 180.0

In [14]:
def Pos2Vec(x: float=0.0, y: float=0.0):
    '''
    Gets the position vector of a point on the 2D-plane
    '''
    return np.array([[x], [y]], dtype=np.float32)

In [15]:
def Pos2Vec3D(x: float=0.0, y: float=0.0, z: float=0.0):
    '''
    Gets the position vector of a point on the 3D-plane
    '''
    return np.array([[z], [y], [x]], dtype=np.float32)

In [16]:
def rotationVector2D(theta: float=np.pi, degrees:bool=True):
    '''
    Gets the rotation vector for theta in radians
    '''
    if degrees: theta *= np.pi / 180
    return np.array(
        [[np.cos(theta), -np.sin(theta)],
        [np.sin(theta), np.cos(theta)]],
        dtype=np.float32
    )

In [17]:
def RotatePoint(point: List[float], angle: float, degrees: bool=True) -> List[float]:
    '''
    Rotates a point (x,y) theta radians counter-clockwise
    '''
    x, y = point
    P = Pos2Vec(x, y)
    R = rotationVector2D(angle, degrees=degrees)
    Q = np.matmul(R, P)
    return Q.reshape((2,))

In [18]:
def Rotate3D(alpha: float=np.pi, beta: float=np.pi, gamma: float=np.pi, degrees: bool=True):
    '''
    3D rotation
    '''
    if degrees: 
        alpha *= np.pi / 180
        beta *= np.pi / 180
        gamma *= np.pi / 180
    return np.array([[np.cos(alpha) * np.cos(beta), np.cos(alpha) * np.sin(beta) * np.sin(gamma) - np.sin(alpha) * np.cos(gamma), np.cos(alpha) * np.sin(beta) * np.cos(gamma) + np.sin(alpha) * np.sin(gamma)],
                    [np.sin(alpha) * np.cos(beta), np.sin(alpha) * np.sin(beta) * np.sin(gamma) + np.cos(alpha) * np.cos(gamma), np.sin(alpha) * np.sin(beta) * np.cos(gamma) - np.cos(alpha) * np.sin(gamma)],
                    [-np.sin(beta), np.cos(beta) * np.sin(gamma), np.cos(beta) * np.cos(gamma)]])

Atom class

In [19]:
# Atom class moved to py file 
%run Atom.py

In [20]:
# database of bond angles basically



⬇️ This class is really really long and it slows down the notebook quite a bit perhaps shove into external file?

In [21]:
molecular_geometries = ["none", "none", "linear", "trigonal planar", "tetrahedral", "trigonal bipyramidal", "octahedral"]

vsepr = [["none", "none", "none", "none", "none", "none", "none"],
         ["none", "none", "none", "none", "none", "none", "none"],
         ["linear", "none", "none", "none", "none", "none", "none"],
         ["trigonal planar", "TP bent", "none", "none", "none", "none", "none"],
         ["tetrahedral", "trigonal pyramidal", "TH bent", "none", "none", "none", "none"],
         ["trigonal bipyramidal", "sawhorse", "T shaped", " TB linear", "none", "none"],
         ["octahedral", "square pyramidal", "square planar", "T shaped", "OC linear", "none", "none"]]

class Molecule:
    def __init__(self, index, adjacency_matrix, atom_symbols):
        self.mapping = {'H': 1, 'C': 4, 'N': 5, 'O': 6, 'F': 7}
        self.index = index
        self.adjacency_matrix = adjacency_matrix
        self.num_atoms = len(self.adjacency_matrix)
        rows, cols = (self.num_atoms, self.num_atoms)
        self.atom_symbols = atom_symbols
        self.atoms = [] 
        
        # traverse_molecule function
        visited_atoms = np.zeros(len(self.atom_symbols), dtype=int)
        stack = []
        start_atom_index = 0
        stack.append(start_atom_index)
        
        visited_atoms[start_atom_index] = 1
        
        # create all the atoms first
        for i in range(len(self.adjacency_matrix)):
            atom = Atom(i, self.atom_symbols[i])
            #atom.calculate_properties(self.atom_symbols[i])
            self.atoms.append(atom)

        # traverse
        while stack:
            current_atom_index = stack.pop()
            current_atom = self.atoms[current_atom_index]
            
            for i in range(len(self.adjacency_matrix)):
                if self.adjacency_matrix[current_atom_index][i] >= 1 and visited_atoms[i] == 0:
                    bond_strength = self.adjacency_matrix[current_atom_index][i]
                    stack.append(i)
                    visited_atoms[i] = 1

                    # Update properties of the current atom
                    atom_current = current_atom
                    atom_current.num_orbitals -= bond_strength
                    atom_current.num_electrons += bond_strength

                    atom_neighboring = self.atoms[i]
                    atom_neighboring.num_orbitals -= bond_strength
                    atom_neighboring.num_electrons += bond_strength
                    
        # octet rule
        self.octet_satisfied = all(atom.num_orbitals == 0 for atom in self.atoms)
        
        # formal charge
        self.formal_charges = [None] * self.num_atoms
        for i in range(len(self.adjacency_matrix)):
            valence_electrons = self.mapping.get(self.atom_symbols[i])
            bonds = sum(self.adjacency_matrix[i])
            if (atom_symbols[i] == 'H'):
                non_bonding_electrons = valence_electrons - (bonds)
            else:
                non_bonding_electrons = valence_electrons - (bonds)
            self.formal_charges[i] = valence_electrons - (bonds + non_bonding_electrons)

        # molecular geometry calculation
        self.lone_pairs = [None] * self.num_atoms
        for i in range(len(self.adjacency_matrix)):
            valence_electrons = self.mapping.get(self.atoms[i].symbol)
            bonds = sum(self.adjacency_matrix[i])
            if (self.atoms[i].symbol == 'H'):
                non_bonding_electrons = valence_electrons - (bonds)
            else:
                non_bonding_electrons = valence_electrons - (bonds)
            self.lone_pairs[i] = int(non_bonding_electrons / 2)
        print(self.lone_pairs)
            
        self.steric_numbers = [None] * self.num_atoms
        self.molecular_geometries = [None] * self.num_atoms
        
        for i in range(len(self.adjacency_matrix)):
            steric = 0
            for j in range(len(self.adjacency_matrix)):
                if self.adjacency_matrix[i][j] != 0:
                    steric += 1
            steric += self.lone_pairs[i]
            self.steric_numbers[i] = steric
        
        for i in range(len(self.adjacency_matrix)):
            self.molecular_geometries[i] = vsepr[self.steric_numbers[i]][self.lone_pairs[i]]
            
        # cyclic test
        queue = deque()
        queue.append(0)
        self.cyclic = False
        visited_atoms = [0] * self.num_atoms
        prev_nodes = [-1] * self.num_atoms
        while len(queue) != 0:
            current_atom_index = queue.popleft()
            if (visited_atoms[current_atom_index] != 1):
                visited_atoms[current_atom_index] = 1
                for i in range(len(self.adjacency_matrix)):
                    if self.adjacency_matrix[current_atom_index][i] >= 1 and visited_atoms[i] == 1 and prev_nodes[current_atom_index] != i:
                        self.cyclic = True
                    if self.adjacency_matrix[current_atom_index][i] >= 1 and visited_atoms[i] == 0:
                        queue.append(i)
                        prev_nodes[i] = current_atom_index

    def add_hydrogens(self, start_atom = 0):
        print()
        print("Running hydrogen addition algorithm ...")
        visited_atoms = []

        stack = []
        start_atom_index = 0
        stack.append(start_atom_index)
        
        while(stack):     
            current_atom_index = stack.pop()
            current_atom = self.atoms[current_atom_index]
            visited_atoms.append(current_atom_index)
            if current_atom.num_orbitals > 0:
                    new_adj_matrix = [[0] * (self.num_atoms + current_atom.num_orbitals) for _ in range(self.num_atoms + current_atom.num_orbitals)] # --> has additional columns to accomodate added hydrogen atoms + each row & column together represents one atom
                    for i in range(0, len(self.adjacency_matrix)):
                        for j in range(0, len(self.adjacency_matrix)):
                            new_adj_matrix[i][j] = self.adjacency_matrix[i][j]
                        
                    original_num_atoms = self.num_atoms     
                    for j in range(original_num_atoms, original_num_atoms + current_atom.num_orbitals):
                        # print(f"adding hydrogen at: {current_atom_index}")
                        new_adj_matrix[current_atom.atom_idx][j] = 1
                        new_adj_matrix[j][current_atom.atom_idx] = 1
                        current_atom.num_orbitals -= 1
                        current_atom.num_electrons += 1
                        self.atom_symbols.append('H')
                        atom = Atom(j, self.atom_symbols[j])
                        self.atoms.append(atom)
                        self.atoms[j].num_orbitals -= 1
                        self.atoms[j].num_electrons += 1

                    # taking out additional empty atoms that don't mean anything
                    new_num_atoms = 0
                    for j in range(0,len(new_adj_matrix)):
                        if (sum(new_adj_matrix[j]) != 0):
                            new_num_atoms += 1
                    self.adjacency_matrix = [[0] * (new_num_atoms) for _ in range(new_num_atoms)]

                    for j in range(0,new_num_atoms):
                        for i in range(0,new_num_atoms):
                            self.adjacency_matrix[j][i] = new_adj_matrix[j][i]
                    self.num_atoms = len(self.adjacency_matrix)
            
            for i in range(len(self.adjacency_matrix)):
                if self.adjacency_matrix[current_atom_index][i] >= 1 and not (i in visited_atoms):
                    stack.append(i)
        
        # recalculation post-hydrogen addition
        # octet rule
        self.octet_satisfied = all(atom.num_orbitals == 0 for atom in self.atoms)
        
        # formal charge
        self.formal_charges = [None] * self.num_atoms
        for i in range(len(self.adjacency_matrix)):
            valence_electrons = self.mapping.get(self.atom_symbols[i])
            bonds = sum(self.adjacency_matrix[i])
            if (self.atom_symbols[i] == 'H'):
                non_bonding_electrons = valence_electrons - (bonds)
            else:
                non_bonding_electrons = valence_electrons - (bonds)
            self.formal_charges[i] = valence_electrons - (bonds + non_bonding_electrons)

        # molecular geometry calculation
        self.lone_pairs = [None] * self.num_atoms
        for i in range(len(self.adjacency_matrix)):
            valence_electrons = self.mapping.get(self.atoms[i].symbol)
            bonds = sum(self.adjacency_matrix[i])
            if (self.atoms[i].symbol == 'H'):
                non_bonding_electrons = valence_electrons - (bonds)
            else:
                non_bonding_electrons = valence_electrons - (bonds)
            self.lone_pairs[i] = int(non_bonding_electrons / 2)
        print(self.lone_pairs)
            
        self.steric_numbers = [None] * self.num_atoms
        self.molecular_geometries = [None] * self.num_atoms
        
        for i in range(len(self.adjacency_matrix)):
            steric = 0
            for j in range(len(self.adjacency_matrix)):
                if self.adjacency_matrix[i][j] != 0:
                    steric += 1
            steric += self.lone_pairs[i]
            self.steric_numbers[i] = steric
        
        for i in range(len(self.adjacency_matrix)):
            self.molecular_geometries[i] = vsepr[self.steric_numbers[i]][self.lone_pairs[i]]
            
        # cyclic test
        queue = deque()
        queue.append(0)
        self.cyclic = False
        visited_atoms = [0] * self.num_atoms
        prev_nodes = [-1] * self.num_atoms
        while len(queue) != 0:
            current_atom_index = queue.popleft()
            if (visited_atoms[current_atom_index] != 1):
                visited_atoms[current_atom_index] = 1
                for i in range(len(self.adjacency_matrix)):
                    if self.adjacency_matrix[current_atom_index][i] >= 1 and visited_atoms[i] == 1 and prev_nodes[current_atom_index] != i:
                        self.cyclic = True
                    if self.adjacency_matrix[current_atom_index][i] >= 1 and visited_atoms[i] == 0:
                        queue.append(i)
                        prev_nodes[i] = current_atom_index
            
        print()
        print(f"updated adjacency matrix: {self.adjacency_matrix}")
        print(f"updated atom symbols: {self.atom_symbols}")
        print(current_atom_index)
        return self.adjacency_matrix
    
    def find_central_atoms(self):
        adjList, symbols = MatToList(self.adjacency_matrix, self.atom_symbols)
        return adjList, symbols
    
#     def create_displacement_coords(self, bond_length, angles, central_coordinates, adj_coordinates, initial_coords=np.array([[0, 0, 0]], level = 0):
# #         # level indicates the recrusion depth the function is at --> when the level is incremented 
# #         # by 1 later on, it means we've moved on to the next bond length & angle in the list
        
#         if level ==len(bond_length):
#                      return initial_coords
            
#         bond_length = bond_length[level]
#         transformation_matrix = Rotate3D(angles, 0, 0)
            
#         applying the transformation to the initial coordinates
#         new_coords = np.matmul(initial_coordinates, transformation_matrix)
#         new_coords[-1, 0] += bond_length
                                   
#         # check if any adjacent atom alreayd has non-zero coordinates 
#         for adj_coord in adj_coordinates:
#                 if not np.array_equal(adj_coord, np.arry([0, 0, 0])):
#                         # do rotation + make the angle between central and adj atom the initial angle
#                         # do matrix multiplication
                                   
#                     # calculates the angle difference between the current bond angle and the initial angle 
#                     # in order to rotate the adj atom around the central atom & make the angle the 'initial' angle               
#                     initial_bond_vector = initial_coords[-1] - central_coordinates
#                     current_bond_vector = adj_coord - central_coordinates
#                     current_angle = np.arccos(np.dot(initial_bond_vector, current_bond_vector) / (np.linalg.norm(initial_bond_vector) * np.linalg.norm(current_bond_vector)))
#                     angle_diff = angle - current_angle
                    
#                     # calculates rotation axis as cross product of the initial & current bond vectors
#                     axis = np.cross(initial_bond_vector, current_bond_vector)
                                   
#                     rotated_adjacent_coords = RotatePoint(adj_coord - central_coordinates, np.degrees(angle_diff))
#                     rotated_adjacent_coords += central_coordinates

#                     # Update the adjacent coordinates
#                     adjacent_coordinates[i] = rotated_adjacent_coords
                                   
#         return self.create_displacement_coords(bond_length, angles, new_coords, level + 1)

# new ver based off pseudocode 3/12/24
    def create_xyz(self, index, prev):
        
        print(f"Performing create_xyz function on index {index}, previous index {prev}")
        print()
        
        nbrs = []
        for i in range(len(self.adjacency_matrix)):
            if(self.adjacency_matrix[index][i] != 0):
                nbrs.append((self.atoms[i], self.adjacency_matrix[index][i])) # atom, bond strength
        print("Adjacent atoms found: ")
        print(nbrs)
        displacementCoords = []
        centralAtom = self.atoms[index]
        
        # now breaking off based on geometry
        if (self.molecular_geometries[index] == "tetrahedral"):
            angle = DegtoRad(109.5) 
            bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((0, 0, BondLengths[bond])) # first angle goes to the right

            bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((BondLengths[bond] * sin(pi - angle), 0, - BondLengths[bond] * cos(pi - angle)))# second angle goes up

            bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , - BondLengths[bond] * sin (angle / 2), BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)))

            bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[3][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle), BondLengths[bond] * sin (angle / 2), BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)))
        if (self.molecular_geometries[index] == "trigonal pyramidal"):
            angle = DegtoRad(107) 
            bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond])) # first angle goes to the right

            bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)))# second angle goes up

            bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
            try: _ = BondLengths[bond]
            except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
            displacementCoords.append((centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y - BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)))
        if (self.molecular_geometries[index] == "trigonal planar"):
            angle = DegtoRad(120.0) #120 bond angle
        #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
        bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
        try: _ = BondLengths[bond]
        except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
        displacementCoords.append((centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]))
        bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
        try: _ = BondLengths[bond]
        except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
        displacementCoords.append((centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)))
        bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
        try: _ = BondLengths[bond]
        except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
        displacementCoords.append((centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)))
        # repeat for other geometries
        
        
        alpha = 0
        beta = 0
        gamma = 0
        
        previ = -1
        for i in range(len(nbrs)):
            if (nbrs[i][0].x != 0 or nbrs[i][0].y != 0 or nbrs[i][0].z != 0 or nbrs[i][0].atom_idx == prev):
                print(f"Atom found with previous connection: {nbrs[i][0].atom_idx}")
                
                u = np.array([displacementCoords[i][0], displacementCoords[i][1], displacementCoords[i][2]])
                v = np.array([nbrs[i][0].x - centralAtom.x, nbrs[i][0].y - centralAtom.y, nbrs[i][0].z] - centralAtom.z)
                print(u)
                print(v)
# #                 # btw i guessed on most of this so if anything is wrong it's this
#                 print((-nbrs[i][0].z+centralAtom.z) / sqrt(1-pow((nbrs[i][0].z-centralAtom.z), 2)))
# #                 alpha = random.randint(0,360)
# #                 beta = random.randint(0, 360)
# #                 gamma = random.randint(0, 360)
# #                 alpha = 0
# #                 beta = 0
# #                 gamma = 0
#                 beta = acos(nbrs[i][0].z-centralAtom.z)
#                 print(beta)
#                 gamma = acos(((nbrs[i][0].y-centralAtom.y) * sin(beta)) / sqrt(1-pow((nbrs[i][0].z-centralAtom.z), 2)))
#                 alpha = acos(((-nbrs[i][0].z+centralAtom.z)* cos(beta)) / sqrt(1-pow((nbrs[i][0].z-centralAtom.z), 2)))
# #                alpha =(180 / pi * atan((nbrs[i][0].y - centralAtom.y)/(nbrs[i][0].x - centralAtom.x))) # rotation around z
# #                beta = (180 / pi * atan((nbrs[i][0].x - centralAtom.x)/(nbrs[i][0].z - centralAtom.z))) # rotation around y
# #                gamma = (180 / pi * atan((nbrs[i][0].z - centralAtom.z)/(nbrs[i][0].y - centralAtom.y))) # rotation around x
#                 previ = i
#                 i-=1
#                 print(f"alpha: {alpha}, beta: {beta}, gamma: {gamma}")
                break
                
        rotationMatrix = Rotate3D(alpha, beta, gamma)
        print("Rotation Matrix:")
        print(rotationMatrix)

        for i in range(len(nbrs)):
            if (previ != i):
                print(f"i = {i}")
                print(f"Assigning coordinates to atom: {nbrs[i][0].atom_idx}")
                vector = Pos2Vec3D(displacementCoords[i][0], displacementCoords[i][1], displacementCoords[i][2])
                print("Vector:")
                print(vector)
                newCoords = np.matmul(rotationMatrix, vector)
                print("new coordinates: ")
                print(newCoords)
                coordList = [newCoords[0][0] + centralAtom.x, newCoords[1][0] + centralAtom.y, newCoords[2][0] + centralAtom.z]
                self.atoms[nbrs[i][0].atom_idx].setCoords(coordList)
                print("xyz so far:")
                print(self.viewXYZ())
                if (self.molecular_geometries[nbrs[i][0].atom_idx] != 'none'):
                    self.create_xyz(nbrs[i][0].atom_idx, index)
        print("backtracking")
        
    def viewXYZ(self):
        for i in range (len(self.atoms)):
             print(self.atoms[i].printXYZ())
    
    def show_properties(self):
        print()
        print("Attributes of this molecule")
        print(f"1) Atom Symbols: {self.atom_symbols}")
        print(f"2) Adjacency Matrix: {self.adjacency_matrix}")
        print(f"3) Octet Rule Satisfied?: {self.octet_satisfied}")
        print(f"4) Formal Charges: {self.formal_charges}")
        print(f"5) Steric Numbers: {self.steric_numbers}")
        print(f"6) Lone Pairs: {self.lone_pairs}")
        print(f"7) VSEPR Molecular Geometries: {self.molecular_geometries}")
        print(f"8) Cyclic: {self.cyclic}")
    
    

In [22]:
atom_symbols5 = ['C', 'H', 'H', 'H', 'C', 'H']
adjacency_matrix5 = np.array([
    [0, 1, 1, 1, 1, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0]])
test_molecule5 = Molecule(index=0, adjacency_matrix=adjacency_matrix5, atom_symbols=atom_symbols5)
test_molecule5.show_properties()
test_molecule5.add_hydrogens()
test_molecule5.show_properties()
test_molecule5.create_xyz(0, -1)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



RecursionError: maximum recursion depth exceeded while calling a Python object

In [ ]:
# example test of displacement coordinates function:

molecule = Molecule(index=0, adjacency_matrix=None, atom_symbols=None)

bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
try: _ = BondLengths[bond]
except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")

bond_lengths = []  
angles = []            # angles in degrees

final_displacement_coords = molecule.displacement_coordinates(bond_lengths, angles)
print("Displacement coordinates:")
print(final_displacements_coords)

### Adjacency Matrix & Geometries Utils

Added some Annotations

In [ ]:
from collections import defaultdict
from GeometryFromAdj import GeometryFromAdj

In [ ]:
def GeometryFromAdj(adjMat: List[List[int]], symbols: List[str]):
    ROBERT = len(adjMat) #too lazy to use N
    adj = {} #maps central atoms to corresponding geometries
    for i, s in enumerate(symbols):
        for j in range(i+1, ROBERT):
            if adjMat[i][j] == 0: continue #no bond so no geometry
            bond = ROBERT - adjMat[i].count(0) #inefficient way of finding number of bonds
            if bond == 1: continue #atom only has one bond so no geometry (not a central atom or is H)
            e_remaining = (4 - sum(adjMat[i])) // 2 #electrons remaining (shouldn't be negative)
            if e_remaining < 0: raise RuntimeError("Electrons are not your brain cells. Electrons cannot disappear.")
            bt = (bond, e_remaining) #get geometry from map defined above
            adj[symbols[i]] = GeometryMap[bt]
    return adj

In [ ]:
adj = [[0, 1, 1, 1, 1],
       [1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0]]
symbols = ["C", "H", "H", "H", "H"]
A = GeometryFromAdj(adj, symbols)

In [ ]:
A

first convert adjacency matrix information to adjacency list because it's easier  
also filter out non-central atoms

In [ ]:
def MatToList(adjMat, symbols):
    adjList = defaultdict(list)
    central = []
    # central = set() #central atoms only
    N = len(adjMat)
    for i, s in enumerate(symbols):
        print(adjMat[i])
        if N - list(adjMat[i]).count(0) == 1: continue #not a central atom, fixed numpy issue
        central.append(s)
        for j in range(N):
            if adjMat[i][j]:
                adjList[i].append(j) #atom i is bonded to atom j
                #central.append(s) #s is a central atom
    return adjList, central

In [ ]:
AL, central = MatToList(adj, symbols)

In [ ]:
central

In [ ]:
AL #works as intended

### Check XYZ Coordinates Against RDKit (Only Works for Molecules that Exist in NIH database)

In [ ]:
# def Mol2SMILES(id):
#     try:
#         url = 'http://cactus.nci.nih.gov/chemical/structure/' + quote(id) + '/smiles'
#         ans = urlopen(url).read().decode('utf8')
#         return ans
#     except:
#         return 'Did not work'

# # idk find a way to easily pull molecule name from molformula w/o manual input

# def XYZChecker(molName):
#     smiles = Mol2SMILES(molName)
#     mol = Chem.MolFromSmiles(smiles)
#     try:
#         mol = Chem.AddHs(mol)
#         AllChem.EmbedMolecule(mol)
#         AllChem.MMFFOptimizeMolecule(mol)
#     except:
#         pass

#     try:
#         coords = mol.GetConformer(0).GetPositions()
#     except:
#         coords = mol.GetConformer().GetPositions()
    
#     return Chem.MolToXYZBlock(mol)

### Linear Geometry, like in $CO_2$

In [ ]:
# def Linear20(centralAtom, nbrs):
#     #nbrs is a list of tuples where each tuple is (Atom, bondOrder)
#     #assume that centralAtom already has a geometry assigned (default is origin)
#     #given centralAtom, calculate coordinates of the two neighbors
#     #linear has 180 degree bond angle
#     #linear geometries confined to a single axis, so randomly select an axis, default is x
#     axis = int(np.random.random() * 3) #(0,x) (1,y) (2,z)
#     bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
#     try: _ = BondLengths[bond] #check if bond exists
#     except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
#     left = [centralAtom.x, centralAtom.y, centralAtom.z]
#     left[axis] -= BondLengths[bond]
#     bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
#     try: _ = BondLengths[bond] #check if bond exists
#     except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
#     right = [centralAtom.x , centralAtom.y, centralAtom.z]
#     right[axis] += BondLengths[bond]
#     nbrs[0][0].setCoords(left)
#     nbrs[1][0].setCoords(right)
#     return nbrs

In [ ]:
centralAtom = Atom(0, "C")
nbrs = [ (Atom(1, "O"), 2), (Atom(2, "O"), 2) ]
nbrs = Linear20(centralAtom, nbrs)

In [ ]:
print(*nbrs, sep='\n')

In [ ]:
print(centralAtom.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Carbon Dioxide'))

In [ ]:


# Doesn't Work
# centralAtom = Atom(0,"C")
# nbrs = [ (Atom(1,"H"), 2), (Atom(2,"H"), 2) ]
# nbrs = Linear20(centralAtom, nbrs)

### Trigonal Planar, like in $NO_3^-$

In [ ]:
def TrigonalPlanar(centralAtom, nbrs):
    angle = DegtoRad(120.0) #120 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [ ]:
center = Atom(0,"N")
nbrs = [(Atom(1,"O"), 2), (Atom(2,"O"), 1), (Atom(3,"O"), 1)]
nbrs = TrigonalPlanar(center, nbrs)

In [ ]:
print(*nbrs, sep="\n")

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Nitrate'))

### Bent with 1 LP, like in $NO_2^-$

In [ ]:
def Bent21(centralAtom, nbrs):
    angle = DegtoRad(117.5) #according to IB Chem, angle is 117.5
    angle /= 2 #since 2-1 bent is just trigonal planar without a vertex, the geometry is still 2D
    #WLOG, assume x-z plane only (easiest to visualize and also I'm lazy)
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x - BondLengths[bond] * sin(angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(angle)]
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(angle)]
    nbrs[0][0].setCoords(left)
    nbrs[1][0].setCoords(right)
    return nbrs

In [ ]:
center = Atom(0, "N")
nbrs = [(Atom(1, "O"), 2), (Atom(2, "O"), 1)]
nbrs = Bent21(center, nbrs)

In [ ]:
print(*nbrs, sep='\n')

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

### Bent with 2 LP, Like $H_2O$

In [ ]:
def Bent22(centralAtom, nbrs):
    angle = DegtoRad(104.5) #according to IB Chem, angle is 117.5
    angle /= 2 #since 2-1 bent is just trigonal planar without a vertex, the geometry is still 2D
    #WLOG, assume x-z plane only (easiest to visualize and also I'm lazy)
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x - BondLengths[bond] * sin(angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(angle)]
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(angle)]
    nbrs[0][0].setCoords(left)
    nbrs[1][0].setCoords(right)
    return nbrs

In [ ]:
center = Atom(0, "O")
nbrs = [(Atom(1, "H"), 1), (Atom(2, "H"), 1)]
nbrs = Bent22(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Water'))

In [ ]:
print(*nbrs, sep='\n')

### Trigonal Pyramidal, Like $NH_3$

In [ ]:
def TrigonalPyramidal(centralAtom, nbrs):
    angle = DegtoRad(107) 
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]] # first angle goes to the right
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]# second angle goes up
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    front = [centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y - BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle) ]

    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(right)
    nbrs[2][0].setCoords(front)
    return nbrs

In [ ]:
center = Atom(0, "N")
nbrs = [(Atom(1, "H"), 1), (Atom(2, "H"), 1), (Atom(3, "H"), 1)]
nbrs = TrigonalPyramidal(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Ammonia'))

In [ ]:
print(*nbrs, sep='\n')

### Tetrahedral, Like $CH_4$

In [ ]:
def Tetrahedral(centralAtom, nbrs):
    angle = DegtoRad(109.5) 
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]] # first angle goes to the right
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]# second angle goes up
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    front = [centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y - BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle) ]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[3][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    back = [centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y + BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)  ]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(right)
    nbrs[2][0].setCoords(front)
    nbrs[3][0].setCoords(back)
    return nbrs

In [ ]:
# returns coordinates as list of tuples instead of modifying the input molecules

def TetrahedralNew (centralAtom, nbrs):
    result = []
    angle = DegtoRad(109.5) 
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    result.append((centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond])) # first angle goes to the right
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    result.append((centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)))# second angle goes up
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[2][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    result.append((centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y - BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)))
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[3][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    result.append((centralAtom.x + BondLengths[bond] * cos (angle / 2) * sin(((2 * pi) - angle) / 2 + angle) , centralAtom.y + BondLengths[bond] * sin (angle / 2), centralAtom.z + BondLengths[bond] * cos(angle / 2) * cos(((2 * pi) - angle) / 2 + angle)))
    return result


In [ ]:
center = Atom(0, "C")
nbrs = [(Atom(1, "H"), 1), (Atom(2, "H"), 1), (Atom(3, "H"), 1), (Atom(4, "H"), 1)]
nbrs = Tetrahedral(center, nbrs)
TetrahedralNew(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Methane'))

In [ ]:
print(*nbrs, sep='\n')

### Trigonal Bipyramidal, Like $PF_5$ (Note: None of the QM9 Molecules have Trigonal Bipyramidal Geometry)

In [ ]:
def TrigonalBipyramidal(centralAtom, nbrs):
    angle = DegtoRad(120)
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z]
    top[1] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottom = [centralAtom.x , centralAtom.y, centralAtom.z]
    bottom[1] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x , centralAtom.y, centralAtom.z]
    left[0] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    backright = [centralAtom.x + BondLengths[bond] * cos(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * sin(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    frontright = [centralAtom.x + BondLengths[bond] * cos(pi - angle), centralAtom.y, centralAtom.z + BondLengths[bond] * sin(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(bottom)
    nbrs[2][0].setCoords(left)
    nbrs[3][0].setCoords(backright)
    nbrs[4][0].setCoords(frontright)
    return nbrs

In [ ]:
center = Atom(0, "P")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1), (Atom(4, "F"), 1), (Atom(5, "F"), 1)]
nbrs = TrigonalBipyramidal(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Phosphorus Pentafluoride'))

### Seesaw, Like $SF_4$

In [ ]:
def Seesaw(centralAtom, nbrs):
    angle = DegtoRad(120)
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z]
    top[1] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottom = [centralAtom.x , centralAtom.y, centralAtom.z]
    bottom[1] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    backright = [centralAtom.x + BondLengths[bond] * cos(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * sin(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    frontright = [centralAtom.x + BondLengths[bond] * cos(pi - angle), centralAtom.y, centralAtom.z + BondLengths[bond] * sin(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(bottom)
    nbrs[2][0].setCoords(backright)
    nbrs[3][0].setCoords(frontright)
    return nbrs

In [ ]:
center = Atom(0, "S")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1), (Atom(4, "F"), 1)]
nbrs = Seesaw(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Sulfur Tetrafluoride')) # This looks like tetrahedral?? maybe the single lone pair doesn't have as big of an effect as we thought??

### T-Shaped, Like $ClF_3$

In [ ]:
def TShaped(centralAtom, nbrs):
    angle = DegtoRad(87.5)
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z]
    top[0] += BondLengths[bond] * cos(pi - angle)
    top[2] += BondLengths[bond] * sin(pi - angle)
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottom = [centralAtom.x , centralAtom.y, centralAtom.z]
    bottom[0] += BondLengths[bond] * cos(pi - angle)
    bottom[2] -= BondLengths[bond] * sin(pi - angle)
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x , centralAtom.y, centralAtom.z]
    left[0] -= BondLengths[bond]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(bottom)
    nbrs[2][0].setCoords(left)
    return nbrs

In [ ]:
center = Atom(0, "Cl")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1)]
nbrs = TShaped(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
# print(XYZChecker('Chlorine Trifluoride')) # breh fix later

### Octahedral, Like $SF_6$

In [ ]:
def Octahedral(centralAtom, nbrs):
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z]
    top[1] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottom = [centralAtom.x , centralAtom.y, centralAtom.z]
    bottom[1] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x , centralAtom.y, centralAtom.z]
    right[0] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x , centralAtom.y, centralAtom.z]
    left[0] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    front = [centralAtom.x , centralAtom.y, centralAtom.z]
    front[2] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[5][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    back = [centralAtom.x , centralAtom.y, centralAtom.z]
    back[2] -= BondLengths[bond]
    
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(bottom)
    nbrs[2][0].setCoords(right)
    nbrs[3][0].setCoords(left)
    nbrs[4][0].setCoords(front)
    nbrs[5][0].setCoords(back)
    
    return nbrs

In [ ]:
center = Atom(0, "S")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1), (Atom(4, "F"), 1), (Atom(5, "F"), 1), (Atom(6, "F"), 1)]
nbrs = Octahedral(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
# print(XYZChecker('Sulfur Hexafluoride')) # breh fix later

### Square Pyramidal, Like $ClF_5$

In [ ]:
def SquarePyramidal(centralAtom, nbrs):
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z]
    top[1] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    front = [centralAtom.x , centralAtom.y, centralAtom.z]
    front[2] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x , centralAtom.y, centralAtom.z]
    left[0] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x , centralAtom.y, centralAtom.z]
    right[0] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    back = [centralAtom.x , centralAtom.y, centralAtom.z]
    back[2] -= BondLengths[bond]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(front)
    nbrs[2][0].setCoords(left)
    nbrs[3][0].setCoords(right)
    nbrs[4][0].setCoords(back)
    return nbrs

In [ ]:
center = Atom(0, "Cl")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1), (Atom(4, "F"), 1), (Atom(5, "F"), 1)]
nbrs = SquarePyramidal(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
# print(XYZChecker('Chlorine Pentafluoride')) # breh fix later

### Square Planar $XeF_4$

In [ ]:
def SquarePlanar(centralAtom, nbrs):
    
    bond = centralAtom.symbol + bondType[nbrs[0][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x , centralAtom.y, centralAtom.z]
    right[0] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x , centralAtom.y, centralAtom.z]
    left[0] -= BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    front = [centralAtom.x , centralAtom.y, centralAtom.z]
    front[2] += BondLengths[bond]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol #multiple bonds allowed
    try: _ = BondLengths[bond] #check if bond exists
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    back = [centralAtom.x , centralAtom.y, centralAtom.z]
    back[2] -= BondLengths[bond]
    
    nbrs[0][0].setCoords(right)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(front)
    nbrs[3][0].setCoords(back)
    
    return nbrs

In [ ]:
center = Atom(0, "Xe")
nbrs = [(Atom(1, "F"), 1), (Atom(2, "F"), 1), (Atom(3, "F"), 1), (Atom(4, "F"), 1)]
nbrs = SquarePlanar(center, nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

In [ ]:
print(XYZChecker('Xenon Tetrafluoride'))

### Cyclic Molecular Geometry

In [ ]:
def Cyclic(nbrs):
    angle = DegtoRad(180)

##### CycloPropane, like C3H6 

In [ ]:
def TriCycle(nbrs): #cyclopropane
    angle = DegtoRad(60) #60 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    #bondType = {1: "-", 2: "=", 3: "#",} 
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    right = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    left = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [ ]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "H"), 1), (Atom(4, "H"), 1), (Atom(5, "H"), 1), (Atom(6, "H"), 1), (Atom(7, "H"), 1), (Atom(8, "H"), 1) ]
nbrs = TriCycle(nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

#### CycloButane, like C4H8

In [49]:
def SquareCycle(nbrs): #cyclobutane
    angle = DegtoRad(90) #90 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topleft = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
   
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [50]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "H"), 1), (Atom(5, "H"), 1), (Atom(6, "H"), 1), (Atom(7, "H"), 1), (Atom(8, "H"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1)]
nbrs = SquareCycle(nbrs)

NameError: name 'centralAtom' is not defined

In [51]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

N 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0


#### CycloPentane, like C5H10

In [ ]:
def PentaCycle(nbrs): #cyclopentane
    angle = DegtoRad(108) #108 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
   
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [ ]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "C"), 1), (Atom(5, "H"), 1), (Atom(6, "H"), 1), (Atom(7, "H"), 1), (Atom(8, "H"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1), (Atom(12, "H"), 1), (Atom(13, "H"), 1), (Atom(14, "H"), 1)]
nbrs = PentaCycle(nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

#### CycloHexane, like C6H12

In [52]:
def HexaCycle(nbrs): #cyclohexane
    angle = DegtoRad(120) #120 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topright = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = nbrs[0][0].symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleleft = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[5][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [53]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "C"), 1), (Atom(5, "C"), 1), (Atom(6, "H"), 1), (Atom(7, "H"), 1), (Atom(8, "H"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1), (Atom(12, "H"), 1), (Atom(13, "H"), 1), (Atom(14, "H"), 1), (Atom(15, "H"), 1), (Atom(16, "H"), 1), (Atom(17, "H"), 1)]
nbrs = HexaCycle(nbrs)

NameError: name 'centralAtom' is not defined

In [54]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

N 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0


#### CycloHeptane, like C7H14

In [ ]:
def HeptaCycle(nbrs): #cycloheptane
    angle = DegtoRad(129) #129 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    upright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    upleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = nbrs[0][0].symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleright = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleleft = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[5][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[6][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [ ]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "C"), 1), (Atom(5, "C"), 1), (Atom(6, "C"), 1), (Atom(7, "H"), 1), (Atom(8, "H"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1), (Atom(12, "H"), 1), (Atom(13, "H"), 1), (Atom(14, "H"), 1), (Atom(15, "H"), 1), (Atom(16, "H"), 1), (Atom(17, "H"), 1), (Atom(18, "H"), 1), (Atom(19, "H"), 1), (Atom(20, "H"), 1)]
nbrs = HeptaCycle(nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

#### CycloOctane, like C8H16

In [ ]:
def OctaCycle(nbrs): #cyclooctane
    angle = DegtoRad(135) #135 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topright = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topleft = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    upleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    upright = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    downleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[5][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    downright = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[6][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[7][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [ ]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "C"), 1), (Atom(5, "C"), 1), (Atom(6, "C"), 1), (Atom(7, "C"), 1), (Atom(8, "H"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1), (Atom(12, "H"), 1), (Atom(13, "H"), 1), (Atom(14, "H"), 1), (Atom(15, "H"), 1), (Atom(16, "H"), 1), (Atom(17, "H"), 1), (Atom(18, "H"), 1), (Atom(19, "H"), 1), (Atom(20, "H"), 1), (Atom(21, "H"), 1), (Atom(22, "H"), 1), (Atom(23, "H"), 1)]
nbrs = OctaCycle(nbrs)

In [ ]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

#### CycloNonane, like C9H18

In [55]:
def NonCycle(nbrs): #cyclononane
    angle = DegtoRad(140) #140 bond angle
    #since geometry is planar, WLOG assume y = 0 so we are in x-z plane only (easiest to visualize)
    
    bond = nbrs[0][0].symbol + bondType[nbrs[0][1]] + nbrs[0][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    top = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[1][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[2][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    topleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = nbrs[0][0].symbol + bondType[nbrs[3][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    undertopleft = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[4][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    undertopright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[5][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = nbrs[0][0].symbol + bondType[nbrs[6][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    middleright = [centralAtom.x, centralAtom.y, centralAtom.z + BondLengths[bond]]
    
    bond = centralAtom.symbol + bondType[nbrs[7][1]] + nbrs[1][0].symbol 
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomright = [centralAtom.x + BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    bond = centralAtom.symbol + bondType[nbrs[8][1]] + nbrs[1][0].symbol
    try: _ = BondLengths[bond]
    except KeyError: raise RuntimeError("This bond is your father, because it doesn't exist.")
    bottomleft = [centralAtom.x - BondLengths[bond] * sin(pi - angle), centralAtom.y, centralAtom.z - BondLengths[bond] * cos(pi - angle)]
    
    nbrs[0][0].setCoords(top)
    nbrs[1][0].setCoords(left)
    nbrs[2][0].setCoords(right)
    return nbrs

In [56]:
nbrs = [(Atom(0, "C"), 1), (Atom(1, "C"), 1), (Atom(2, "C"), 1), (Atom(3, "C"), 1), (Atom(4, "C"), 1), (Atom(5, "C"), 1), (Atom(6, "C"), 1), (Atom(7, "C"), 1), (Atom(8, "C"), 1), (Atom(9, "H"), 1), (Atom(10, "H"), 1), (Atom(11, "H"), 1), (Atom(12, "H"), 1), (Atom(13, "H"), 1), (Atom(14, "H"), 1), (Atom(15, "H"), 1), (Atom(16, "H"), 1), (Atom(17, "H"), 1), (Atom(18, "H"), 1), (Atom(19, "H"), 1), (Atom(20, "H"), 1), (Atom(21, "H"), 1), (Atom(22, "H"), 1), (Atom(23, "H"), 1), (Atom(24, "H"), 1), (Atom(25, "H"), 1), (Atom(26, "H"), 1)]
nbrs = NonCycle(nbrs)

NameError: name 'centralAtom' is not defined

In [57]:
print(center.printXYZ())
for i in range (len(nbrs)):
    print(nbrs[i][0].printXYZ())

N 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
C 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0
H 0.0 0.0 0.0


### 3D Distance Formula for Testing

In [ ]:
D = lambda p1, p2: sqrt(
    pow(p1[0]-p2[0], 2) +
    pow(p1[1]-p2[1], 2) +
    pow(p1[2]-p2[2], 2)
)

### Molecule Testing

In [23]:
adjacency_matrix1 = np.array([
     [0, 1, 1],
     [1, 0, 0],
     [1, 0, 0]])
molecule_symbols1 =  ['C', 'H', 'H']
test_molecule = Molecule(index=0, adjacency_matrix=adjacency_matrix1, atom_symbols=molecule_symbols1)
test_molecule.show_properties()
test_molecule.add_hydrogens()
test_molecule.show_properties()
test_molecule.find_central_atoms()
test_molecule.create_xyz(0, -1)
test_molecule.viewXYZ()

[1, 0, 0]

Attributes of this molecule
1) Atom Symbols: ['C', 'H', 'H']
2) Adjacency Matrix: [[0 1 1]
 [1 0 0]
 [1 0 0]]
3) Octet Rule Satisfied?: False
4) Formal Charges: [0, 0, 0]
5) Steric Numbers: [3, 1, 1]
6) Lone Pairs: [1, 0, 0]
7) VSEPR Molecular Geometries: ['TP bent', 'none', 'none']
8) Cyclic: False

Running hydrogen addition algorithm ...
[0, 0, 0, 0, 0]

updated adjacency matrix: [[0, 1, 1, 1, 1], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0]]
updated atom symbols: ['C', 'H', 'H', 'H', 'H']
4

Attributes of this molecule
1) Atom Symbols: ['C', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0, 1, 1, 1, 1], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0]]
3) Octet Rule Satisfied?: True
4) Formal Charges: [0, 0, 0, 0, 0]
5) Steric Numbers: [4, 1, 1, 1, 1]
6) Lone Pairs: [0, 0, 0, 0, 0]
7) VSEPR Molecular Geometries: ['tetrahedral', 'none', 'none', 'none', 'none']
8) Cyclic: False


NameError: name 'MatToList' is not defined

In [24]:
atom_num2 = [6, 6, 1, 1, 1, 1]
adj_matrixx2 = ([
    [0, 2, 1, 1, 0, 0],
    [2, 0, 0, 0, 1, 1],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0]])

#print(MatToList(adj_matrixx2, atom_symbols2))

atom_symbols2 =  ['C', 'C', 'H', 'H', 'H', 'H']
test_molecule2 = Molecule(index=0, adjacency_matrix=adj_matrixx2, atom_symbols=atom_symbols2)
test_molecule2.show_properties()
test_molecule2.add_hydrogens()
test_molecule2.show_properties()
test_molecule2.find_central_atoms()
test_molecule2.create_xyz(0, -1)

[0, 0, 0, 0, 0, 0]

Attributes of this molecule
1) Atom Symbols: ['C', 'C', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0, 2, 1, 1, 0, 0], [2, 0, 0, 0, 1, 1], [1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0]]
3) Octet Rule Satisfied?: True
4) Formal Charges: [0, 0, 0, 0, 0, 0]
5) Steric Numbers: [3, 3, 1, 1, 1, 1]
6) Lone Pairs: [0, 0, 0, 0, 0, 0]
7) VSEPR Molecular Geometries: ['trigonal planar', 'trigonal planar', 'none', 'none', 'none', 'none']
8) Cyclic: False

Running hydrogen addition algorithm ...
[0, 0, 0, 0, 0, 0]

updated adjacency matrix: [[0, 2, 1, 1, 0, 0], [2, 0, 0, 0, 1, 1], [1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0]]
updated atom symbols: ['C', 'C', 'H', 'H', 'H', 'H']
5

Attributes of this molecule
1) Atom Symbols: ['C', 'C', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0, 2, 1, 1, 0, 0], [2, 0, 0, 0, 1, 1], [1, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0]]
3) Octet Rule Satisf

NameError: name 'MatToList' is not defined

In [25]:
atom_symbols10 = ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']

adjacency_matrix10 = ( 
[[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
 [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

test_molecule10 = Molecule(index=0, adjacency_matrix=adjacency_matrix10, atom_symbols=atom_symbols10)
test_molecule10.show_properties()
test_molecule10.add_hydrogens()
test_molecule10.show_properties()
test_molecule10.find_central_atoms()
test_molecule10.create_xyz(0, -1)

[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attributes of this molecule
1) Atom Symbols: ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0], [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1

NameError: name 'MatToList' is not defined

In [26]:
atom_num2 = [6, 6, 1, 1, 1, 1]
adj_matrixx2 = np.array([
    [0, 2, 1, 1, 0, 0],
    [2, 0, 0, 0, 1, 1],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0]])
atom_symbols2 =  ['C', 'C', 'H', 'H', 'H', 'H']
test_molecule2 = Molecule(index=0, adjacency_matrix=adj_matrixx2, atom_symbols=atom_symbols2)
test_molecule2.show_properties()
test_molecule2.add_hydrogens()
test_molecule2.show_properties()
test_molecule2.create_xyz(0, -1)

[0, 0, 0, 0, 0, 0]

Attributes of this molecule
1) Atom Symbols: ['C', 'C', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0 2 1 1 0 0]
 [2 0 0 0 1 1]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 1 0 0 0 0]]
3) Octet Rule Satisfied?: True
4) Formal Charges: [0, 0, 0, 0, 0, 0]
5) Steric Numbers: [3, 3, 1, 1, 1, 1]
6) Lone Pairs: [0, 0, 0, 0, 0, 0]
7) VSEPR Molecular Geometries: ['trigonal planar', 'trigonal planar', 'none', 'none', 'none', 'none']
8) Cyclic: False

Running hydrogen addition algorithm ...
[0, 0, 0, 0, 0, 0]

updated adjacency matrix: [[0 2 1 1 0 0]
 [2 0 0 0 1 1]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 1 0 0 0 0]]
updated atom symbols: ['C', 'C', 'H', 'H', 'H', 'H']
5

Attributes of this molecule
1) Atom Symbols: ['C', 'C', 'H', 'H', 'H', 'H']
2) Adjacency Matrix: [[0 2 1 1 0 0]
 [2 0 0 0 1 1]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 1 0 0 0 0]]
3) Octet Rule Satisfied?: True
4) Formal Charges: [0, 0, 0, 0, 0, 0]
5) Steric Numbers: [3, 3, 1, 1, 1, 1]
6) 

/tmp/ipykernel_751154/3284377406.py:381: RuntimeWarning: invalid value encountered in matmul
  newCoords = np.matmul(rotationMatrix, vector)


RecursionError: maximum recursion depth exceeded while calling a Python object

In [ ]:
atom_symbols3 = ['N', 'H', 'H']
adjacency_matrix3 = np.array([
     [0, 1, 1],
     [1, 0, 0],
     [1, 0, 0]])
test_molecule3 = Molecule(index=0, adjacency_matrix=adjacency_matrix3, atom_symbols=atom_symbols3)
test_molecule3.show_properties()
test_molecule3.add_hydrogens()
test_molecule3.show_properties()
test_molecule3.create_xyz(0, -1)

In [ ]:
atom_symbols4 = ['C', 'C', 'H', 'H', 'H']
adjacency_matrix4 = np.array([
    [0, 2, 1, 1, 0],
    [2, 0, 0, 0, 1],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0]])
test_molecule4 = Molecule(index=0, adjacency_matrix=adjacency_matrix4, atom_symbols=atom_symbols4)
test_molecule4.show_properties()
test_molecule4.add_hydrogens()
test_molecule4.show_properties()
test_molecule4.create_xyz(0, -1)

In [ ]:
atom_symbols5 = ['C', 'H', 'H', 'H', 'C', 'H']
adjacency_matrix5 = np.array([
    [0, 1, 1, 1, 1, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0]])
test_molecule5 = Molecule(index=0, adjacency_matrix=adjacency_matrix5, atom_symbols=atom_symbols5)
test_molecule5.show_properties()
test_molecule5.add_hydrogens()
test_molecule5.show_properties()
test_molecule5.create_xyz(0, -1)

In [ ]:
test_molecule5.viewXYZ()

In [ ]:
atom_symbols6 = ['C', 'C']
adjacency_matrix6 = np.array([
    [0, 1],
    [1, 0]])
test_molecule6 = Molecule(index=0, adjacency_matrix=adjacency_matrix6, atom_symbols=atom_symbols6)
test_molecule6.show_properties()
test_molecule6.add_hydrogens()
test_molecule6.show_properties()
test_molecule6.create_xyz(0, -1)

In [27]:
atom_symbols7 = ['F', 'C', 'F', 'F', 'C', 'C', 'F', 'F', 'F', 'H', 'H']
adjacency_matrix7 = np.array([
     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
     [1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
     [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
     [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1],
     [0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0],
     [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]]
)
test_molecule7 = Molecule(index=0, adjacency_matrix=adjacency_matrix7, atom_symbols=atom_symbols7)
test_molecule7.show_properties()
test_molecule7.add_hydrogens()
test_molecule7.show_properties()
test_molecule7.find_central_atoms()
test_molecule7.create_xyz(1, -1)
test_molecule7.viewXYZ()

[3, 0, 3, 3, 0, 0, 3, 3, 3, 0, 0]

Attributes of this molecule
1) Atom Symbols: ['F', 'C', 'F', 'F', 'C', 'C', 'F', 'F', 'F', 'H', 'H']
2) Adjacency Matrix: [[0 1 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 1 0 1 1 1 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0]]
3) Octet Rule Satisfied?: True
4) Formal Charges: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
5) Steric Numbers: [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1]
6) Lone Pairs: [3, 0, 3, 3, 0, 0, 3, 3, 3, 0, 0]
7) VSEPR Molecular Geometries: ['none', 'tetrahedral', 'none', 'none', 'tetrahedral', 'tetrahedral', 'none', 'none', 'none', 'none', 'none']
8) Cyclic: False

Running hydrogen addition algorithm ...
[3, 0, 3, 3, 0, 0, 3, 3, 3, 0, 0]

updated adjacency matrix: [[0 1 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 1

NameError: name 'MatToList' is not defined

In [ ]:
atom_symbols8 = ['C', 'O', 'C', 'H', 'H', 'H', 'H', 'H', 'H']
adjacency_matrix8 = np.array(
    [[0, 1, 0, 1, 1, 1, 0, 0, 0],
     [1, 0, 1, 0, 0, 0, 0, 0, 0],
     [0, 1, 0, 0, 0, 0, 1, 1, 1],
     [1, 0, 0, 0, 0, 0, 0, 0, 0],
     [1, 0, 0, 0, 0, 0, 0, 0, 0],
     [1, 0, 0, 0, 0, 0, 0, 0, 0],
     [0, 0, 1, 0, 0, 0, 0, 0, 0],
     [0, 0, 1, 0, 0, 0, 0, 0, 0],
     [0, 0, 1, 0, 0, 0, 0, 0, 0]])

test_molecule8 = Molecule(index=0, adjacency_matrix=adjacency_matrix8, atom_symbols=atom_symbols8)
test_molecule8.show_properties()
test_molecule8.add_hydrogens()
test_molecule8.show_properties()
test_molecule8.find_central_atoms()

In [28]:
atom_symbols9 = ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']

adjacency_matrix9 = np.array([[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
 [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

test_molecule9 = Molecule(index=0, adjacency_matrix=adjacency_matrix9, atom_symbols=atom_symbols9)
test_molecule9.show_properties()
test_molecule9.add_hydrogens()
test_molecule9.show_properties()
test_molecule9.create_xyz(0, -1)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



RecursionError: maximum recursion depth exceeded while calling a Python object

In [ ]:
atom_symbols10 = ['C', 'C', 'H', 'H', 'H', 'H']

adjacency_matrix10 = np.array([[0, 1, 1, 1, 0, 0],
[1, 0, 0, 0, 1, 1],
[1, 0, 0, 0, 0, 0],
[1, 0, 0, 0, 0, 0],
[0, 1, 0, 0, 0, 0],
[0, 1, 0, 0, 0, 0]])
                    

test_molecule10 = Molecule(index=0, adjacency_matrix=adjacency_matrix10, atom_symbols=atom_symbols10)
test_molecule10.show_properties()
test_molecule10.add_hydrogens()
test_molecule10.show_properties()
test_molecule10.find_central_atoms()
test_molecule10.create_xyz(0, -1)
test_molecule10.viewXYZ()

In [ ]:
atom_symbols11 = ['H', 'H', 'O']

adjacency_matrix11 = np.array([[0, 0, 1],
                               [0, 0, 1],
                               [1, 1, 0]])
                    

test_molecule11 = Molecule(index=0, adjacency_matrix=adjacency_matrix11, atom_symbols=atom_symbols11)
test_molecule11.show_properties()
test_molecule11.add_hydrogens()
test_molecule11.show_properties()

In [ ]:
atom_symbols12 = ['F', 'F']

adjacency_matrix12 = np.array([[0, 1],
                               [1, 0]])
                    

test_molecule12 = Molecule(index=0, adjacency_matrix=adjacency_matrix12, atom_symbols=atom_symbols12)
test_molecule12.show_properties()
test_molecule12.add_hydrogens()
test_molecule12.show_properties()

In [ ]:
atom_symbols11 = ['O', 'C', 'O', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H','H', 'H']

adjacency_matrix11 = np.array(
[[0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]])
                    

test_molecule11 = Molecule(index=0, adjacency_matrix=adjacency_matrix11, atom_symbols=atom_symbols11)
test_molecule11.show_properties()

In [ ]:
QM9288_sym = ['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']

QM9288_adj = np.array(
[[0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
 [0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
                    

QM9288 = Molecule(index=0, adjacency_matrix=QM9288_adj, atom_symbols=QM9288_sym)
QM9288.show_properties()
